# ErgoStudy AI final end-to-end demo

This notebook exercises the real local FastAPI pipeline, then presents the saved Stage 9 evaluation. Planning and sensor decisions remain deterministic; the local language model is optional.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
inputs = json.loads((root / 'data' / 'processed' / 'final_demo_inputs.json').read_text(encoding='utf-8'))
scenario_by_id = {item['id']: item for item in inputs['scenarios']}
print(f"Loaded {len(scenario_by_id)} competition scenarios for {inputs['project_version']}.")

## Live deterministic path

The application starts with the cached MiniLM model and Chroma collection. This cell sends a complete sensor-aware request without calling Ollama.

In [ ]:
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from fastapi.testclient import TestClient
from api.main import create_app
from api.settings import APISettings

body = scenario_by_id['long_sitting_adaptation']['request']
with TestClient(create_app(settings=APISettings.from_environment(root))) as client:
    readiness = client.get('/api/v1/readiness')
    response = client.post('/api/v1/plans/full', json=body)
assert readiness.status_code == 200 and readiness.json()['ready']
assert response.status_code == 200
live = response.json()
assert live['adapted_plan']['total_planned_minutes'] <= live['adapted_plan']['total_available_minutes']
print('Deterministic services ready:', readiness.json()['ready'])
print('Plan ID:', live['original_plan']['plan_id'])
print('Scheduled subjects:', [item['subject'] for item in live['original_plan']['scheduled_subjects']])
print('Sensor triggers:', live['adapted_plan']['triggers'])
print('Final planned minutes:', live['adapted_plan']['total_planned_minutes'])

## Preserved evaluation and performance evidence

The complete outputs are committed as JSON so the presentation does not depend on a slow optional generation request.

In [ ]:
evaluation = json.loads((root / 'data' / 'processed' / 'final_evaluation_results.json').read_text(encoding='utf-8'))
performance = json.loads((root / 'data' / 'processed' / 'final_performance_results.json').read_text(encoding='utf-8'))
assert evaluation['failed'] == 0
assert performance['concurrent_batch']['successful_requests'] == 5
print(f"Scenarios: {evaluation['passed']}/{evaluation['scenario_count']} passed")
for name, stats in performance['warm_endpoints'].items():
    print(f"{name}: average {stats['average_ms']:.3f} ms, p95 {stats['p95_ms']:.3f} ms")
print('Concurrent batch elapsed:', performance['concurrent_batch']['batch_elapsed_ms'], 'ms')

In [ ]:
outputs = json.loads((root / 'data' / 'processed' / 'final_demo_outputs.json').read_text(encoding='utf-8'))
ollama = next(item for item in outputs['outputs'] if item['id'] == 'ollama_available_explanation')
timeout = next(item for item in outputs['outputs'] if item['id'] == 'ollama_timeout_fallback')
print('Real Ollama mode:', ollama['response']['explanation']['generation_mode'])
print('Timeout fallback:', timeout['response']['fallback_reason_code'])
assert timeout['response']['generation_mode'] == 'deterministic_fallback'
print('The deterministic plan remains valid in both cases.')